# LLaMA-2-7B Per-layer Partial-Sum Exponent Profiling (baseline BFP)

Characterize the concentration of group-sized **partial-sum** exponents on the
**baseline BFP** path (no OB-Skip, no Triton). For every quantized `nn.Linear`, a forward
pre-hook quantizes the activation exactly like `bfp.ipynb`, forms contiguous group-sized partial
dot products in FP32 against the already-BFP-quantized weight, and histograms
`floor(log2(|partial|))` per layer. One run sweeps **BFP8 through BFP4** using the
configured `BFP.block_size`,
using the same uniformly sampled dataset blocks for every format.

This analysis tests whether most partial-sum exponents fall inside a compact interval while
only a small fraction forms sparse tails. The raw per-layer histograms are retained in JSON;
coverage widths such as W90, W95, and W99, plus all figures, are derived later by a separate
local plotting script. The raw distribution is **independent of the OB-Skip threshold T**, so
there is no threshold sweep here. See `docs/Process.md` for the project roadmap and
evaluation rules.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import os
import platform
import re
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
MANTISSA_BITS_SWEEP = (7, 6, 5, 4, 3)  # BFP8, BFP7, BFP6, BFP5, BFP4.

# --- Profiling knobs (this analysis is baseline BFP, T-independent; no OB-Skip, no Triton) ---
MAX_PROFILE_BLOCKS = 8   # sampled uniformly across all complete 2048-token blocks
TOKEN_STRIDE = 1         # >1 subsamples tokens within each block to cut cost
OUT_SUBSAMPLE = 1024     # evenly spaced output channels per layer (None = all channels)
GROUP_CHUNK = 16         # groups per batched matmul (bounds transient GPU memory)
EXP_BIN_MIN = -126       # floor(log2(abs(x))) after the same FP32-tiny clamp as OB-Skip
EXP_BIN_MAX = 127        # complete normal FP32 exponent range; guard bins remain for validation


@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 32
    shared_exponent_bits: int = 5
    mantissa_bits: int = 7  # Excludes the sign bit: 1S7M.
    rounding: str = "nearest"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = True

    def validate(self):
        if self.block_size <= 0:
            raise ValueError("block_size must be positive.")
        if self.shared_exponent_bits < 2:
            raise ValueError("shared_exponent_bits must be at least 2.")
        if self.mantissa_bits <= 0:
            raise ValueError("mantissa_bits must be positive.")
        if self.rounding not in {"nearest", "trunc"}:
            raise ValueError("rounding must be 'nearest' or 'trunc'.")


BFP = BFPConfig()
BFP.validate()
if MAX_PROFILE_BLOCKS <= 0 or TOKEN_STRIDE <= 0 or GROUP_CHUNK <= 0:
    raise ValueError("Profiling counts and strides must be positive.")
if OUT_SUBSAMPLE is not None and OUT_SUBSAMPLE <= 0:
    raise ValueError("OUT_SUBSAMPLE must be positive or None.")
if EXP_BIN_MIN > EXP_BIN_MAX:
    raise ValueError("EXP_BIN_MIN must not exceed EXP_BIN_MAX.")
OUTPUT_DIR = Path("exponent-profile-results")
ARCHIVE_PATH = Path("llama2-7b-exponent-profile.zip")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"Base config: {BFP}")
print(f"Sweep: {[f'BFP{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")
print(
    f"Profiling blocks: {MAX_PROFILE_BLOCKS}, token stride: {TOKEN_STRIDE}, "
    f"out subsample: {OUT_SUBSAMPLE}, exponent bins: [{EXP_BIN_MIN}, {EXP_BIN_MAX}]"
)

## BFP convention (identical to `bfp.ipynb`)

Blocks are contiguous along the last tensor dimension. Each weight row and each token
activation vector is partitioned independently. For a block maximum magnitude `a`, the shared
exponent is `floor(log2(a))`, clamped to the signed E-bit range. With M magnitude bits, the
quantization step is `2 ** (shared_exp - (M - 1))`; mantissas use the symmetric integer range
`[-(2**M - 1), +(2**M - 1)]`. The helpers below are reused verbatim from `bfp.ipynb`.

In [ ]:
def _quantize_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    shared_exp = shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(max_abs == 0, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa = blocks / step
    mantissa = torch.round(mantissa) if config.rounding == "nearest" else torch.trunc(mantissa)
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = mantissa.clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape)
    return dequantized.to(rows.dtype)


def quantize_bfp(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config)

    output = torch.empty_like(flat)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        output[start:end] = _quantize_bfp_rows(flat[start:end], config)

    return output.reshape_as(tensor)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        weight[start:end].copy_(_quantize_bfp_rows(weight[start:end], config))


class BFPLinear(nn.Module):
    def __init__(self, linear, config):
        super().__init__()
        self.linear = linear
        self.config = config

    def forward(self, x):
        x_bfp = quantize_bfp(x, self.config, self.config.activation_chunk_rows)
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = []

    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name

        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            quantize_weight_in_place(child.weight, config)
            setattr(module, name, BFPLinear(child, config))
            replaced.append(full_name)
        else:
            replaced.extend(replace_linear_layers(child, config, full_name))

    return replaced


sample = torch.tensor([[0.0, -1.0, 0.5, 1.5]], device="cuda", dtype=torch.float16)
sample_q = _quantize_bfp_rows(sample, BFP)
assert sample_q.shape == sample.shape
assert sample_q.dtype == torch.float16
assert torch.isfinite(sample_q).all()

## Partial-sum exponent profiler

`ExponentProfiler` registers a forward pre-hook on each `BFPLinear`. The hook re-derives the
group-sized partials only for statistics; it does not change the layer output. Partitioning and
zero-padding of the reduction (K) dimension match `_quantize_bfp_rows`. Counters are kept per
layer (keyed by module name) plus zero / out-of-range tallies, so nothing pollutes the bins.

In [ ]:
def parse_layer_meta(name):
    match = re.search(r"layers\.(\d+)\.", name)
    layer_index = int(match.group(1)) if match else -1
    leaf = name.split(".")[-1]
    return leaf, layer_index


def evenly_spaced_indices(total, count, device=None):
    if total <= 0:
        raise ValueError("total must be positive.")
    if count is None or count >= total:
        return torch.arange(total, device=device, dtype=torch.long)
    if count <= 0:
        raise ValueError("count must be positive or None.")
    if count == 1:
        return torch.tensor([total // 2], device=device, dtype=torch.long)
    return torch.linspace(0, total - 1, steps=count, device=device).round().to(torch.long)


class ExponentProfiler:
    # Collect per-layer histograms of group-sized partial-sum base-2 exponents.
    #
    # Registers a forward pre-hook on every BFPLinear. For each layer it quantizes
    # the activation exactly like BFPLinear, forms contiguous group-sized partial dot
    # products in FP32 against the already-BFP-quantized weight, and histograms
    # floor(log2(|partial|)). Output channels are sampled at evenly spaced indices,
    # and their original/profiled counts are retained for reproducibility. This is
    # the baseline BFP path and is independent of the OB-Skip threshold T.

    def __init__(self, config, bin_min, bin_max, token_stride=1, out_subsample=None, group_chunk=32):
        self.config = config
        self.bin_min = bin_min
        self.bin_max = bin_max
        self.nbins = bin_max - bin_min + 1
        self.token_stride = max(1, int(token_stride))
        self.out_subsample = out_subsample
        self.group_chunk = max(1, int(group_chunk))
        self.layers = {}
        self.handles = []
        self._tiny = torch.finfo(torch.float32).tiny

    def _ensure(self, name, device, input_features, total_out, profiled_out, num_groups):
        shape_meta = (input_features, total_out, profiled_out, num_groups)
        if name not in self.layers:
            self.layers[name] = {
                "counts": torch.zeros(self.nbins, dtype=torch.long, device=device),
                "num_zero": torch.zeros((), dtype=torch.long, device=device),
                "num_underflow": torch.zeros((), dtype=torch.long, device=device),
                "num_overflow": torch.zeros((), dtype=torch.long, device=device),
                "total": torch.zeros((), dtype=torch.long, device=device),
                "observed_min": torch.full((), float("inf"), device=device),
                "observed_max": torch.full((), float("-inf"), device=device),
                "shape_meta": shape_meta,
                "num_calls": 0,
                "profiled_tokens": 0,
            }
        acc = self.layers[name]
        if acc["shape_meta"] != shape_meta:
            raise RuntimeError(f"Layer shape changed while profiling {name}.")
        acc["num_calls"] += 1
        return acc

    @torch.no_grad()
    def _update(self, name, x, weight):
        config = self.config
        block = config.block_size
        width = x.shape[-1]
        flat = x.reshape(-1, width)
        if self.token_stride > 1:
            flat = flat[:: self.token_stride]
        xq = quantize_bfp(flat, config, config.activation_chunk_rows).float()
        weight_view = weight.detach()
        total_out = weight_view.shape[0]
        if self.out_subsample is not None and total_out > self.out_subsample:
            indices = evenly_spaced_indices(total_out, self.out_subsample, weight_view.device)
            weight_view = weight_view.index_select(0, indices)
        wq = weight_view.float()
        pad = (-width) % block
        if pad:
            xq = F.pad(xq, (0, pad))
            wq = F.pad(wq, (0, pad))
        num_tokens = xq.shape[0]
        num_out = wq.shape[0]
        num_groups = xq.shape[1] // block
        acc = self._ensure(name, xq.device, width, total_out, num_out, num_groups)
        acc["profiled_tokens"] += num_tokens
        # [G, M, block] and [G, block, O] so each group is one batched matmul.
        xq_g = xq.reshape(num_tokens, num_groups, block).permute(1, 0, 2).contiguous()
        wq_g = wq.reshape(num_out, num_groups, block).permute(1, 2, 0).contiguous()
        for start in range(0, num_groups, self.group_chunk):
            end = min(start + self.group_chunk, num_groups)
            partials = torch.bmm(xq_g[start:end], wq_g[start:end])  # [g, M, O]
            self._accumulate(acc, partials)

    def _accumulate(self, acc, partials):
        absp = partials.abs()
        zero_mask = absp == 0
        num_zero = zero_mask.sum()
        acc["total"] += partials.numel()
        acc["num_zero"] += num_zero
        exponent = torch.floor(torch.log2(absp.clamp_min(self._tiny)))
        exponent = torch.where(zero_mask, self.bin_min - 1.0, exponent)
        min_candidate = torch.where(zero_mask, float("inf"), exponent).amin()
        max_candidate = torch.where(zero_mask, float("-inf"), exponent).amax()
        acc["observed_min"] = torch.minimum(acc["observed_min"], min_candidate)
        acc["observed_max"] = torch.maximum(acc["observed_max"], max_candidate)
        # Two guard bins retain validation counts without clipping the main histogram.
        offset = exponent.clamp(self.bin_min - 1, self.bin_max + 1).to(torch.long) - (self.bin_min - 1)
        hist = torch.bincount(offset.reshape(-1), minlength=self.nbins + 2)
        acc["counts"] += hist[1 : self.nbins + 1]
        acc["num_overflow"] += hist[self.nbins + 1]
        acc["num_underflow"] += hist[0] - num_zero

    def attach(self, model):
        for name, module in model.named_modules():
            if isinstance(module, BFPLinear):
                self.handles.append(
                    module.register_forward_pre_hook(self._make_hook(name))
                )
        return self

    def _make_hook(self, name):
        def hook(module, args):
            self._update(name, args[0], module.linear.weight)
            return None
        return hook

    def remove(self):
        for handle in self.handles:
            handle.remove()
        self.handles = []

    def export(self):
        records = []
        for name, acc in self.layers.items():
            leaf, index = parse_layer_meta(name)
            input_features, total_out, profiled_out, num_groups = acc["shape_meta"]
            counts = acc["counts"].tolist()
            num_zero = int(acc["num_zero"].item())
            num_underflow = int(acc["num_underflow"].item())
            num_overflow = int(acc["num_overflow"].item())
            total = int(acc["total"].item())
            if sum(counts) + num_zero + num_underflow + num_overflow != total:
                raise RuntimeError(f"Histogram accounting mismatch for {name}.")
            nonzero = total - num_zero
            records.append({
                "layer_name": name,
                "layer_type": leaf,
                "layer_index": index,
                "input_features": input_features,
                "total_output_channels": total_out,
                "profiled_output_channels": profiled_out,
                "num_groups": num_groups,
                "num_calls": acc["num_calls"],
                "profiled_tokens": acc["profiled_tokens"],
                "counts": counts,
                "num_zero_partials": num_zero,
                "num_underflow": num_underflow,
                "num_overflow": num_overflow,
                "total_partials": total,
                "observed_min_exponent": int(acc["observed_min"].item()) if nonzero else None,
                "observed_max_exponent": int(acc["observed_max"].item()) if nonzero else None,
            })
        records.sort(
            key=lambda record: (record["layer_index"] < 0, record["layer_index"], record["layer_type"])
        )
        return records

In [ ]:
def _reference_counts(x, weight, config, bin_min, bin_max):
    # Independent-ish recomputation used only as an integration smoke test.
    width = x.shape[-1]
    xq = quantize_bfp(x.reshape(-1, width), config, config.activation_chunk_rows).float()
    wq = weight.detach().float()
    pad = (-width) % config.block_size
    if pad:
        xq = F.pad(xq, (0, pad))
        wq = F.pad(wq, (0, pad))
    num_groups = xq.shape[1] // config.block_size
    xq_g = xq.reshape(xq.shape[0], num_groups, config.block_size)
    wq_g = wq.reshape(wq.shape[0], num_groups, config.block_size)
    tiny = torch.finfo(torch.float32).tiny
    counts = torch.zeros(bin_max - bin_min + 1, dtype=torch.long, device=x.device)
    for g in range(num_groups):
        partial = xq_g[:, g, :] @ wq_g[:, g, :].t()
        vals = partial.abs()[partial.abs() > 0]
        exponent = torch.floor(torch.log2(vals.clamp_min(tiny)))
        keep = (exponent >= bin_min) & (exponent <= bin_max)
        counts += torch.bincount(exponent[keep].to(torch.long) - bin_min, minlength=counts.numel())
    return counts


torch.manual_seed(0)
_K, _out, _M = 70, 37, 7  # Exercises K-padding and multiple group chunks.
_lin = nn.Linear(_K, _out, bias=False).cuda().half()
_cfg = replace(BFP, mantissa_bits=7)
quantize_weight_in_place(_lin.weight, _cfg)
_x = torch.randn(1, _M, _K, device="cuda", dtype=torch.float16)
_x[:, 0, :32] = 0  # Exercises explicit zero-partial accounting.

_prof = ExponentProfiler(_cfg, EXP_BIN_MIN, EXP_BIN_MAX, group_chunk=1)
_prof._update("test", _x, _lin.weight)
_ref = _reference_counts(_x, _lin.weight, _cfg, EXP_BIN_MIN, EXP_BIN_MAX)
assert torch.equal(_prof.layers["test"]["counts"], _ref), "profiler counts mismatch"
_record = _prof.export()[0]
_expected_total = _M * _out * ((_K + _cfg.block_size - 1) // _cfg.block_size)
assert _record["total_partials"] == _expected_total
assert _record["num_zero_partials"] >= _out
_indices = evenly_spaced_indices(_out, 11, device="cuda")
assert _indices.numel() == 11 and _indices[0] == 0 and _indices[-1] == _out - 1
assert torch.unique(_indices).numel() == _indices.numel()
print("Profiler sanity check passed:", _record["total_partials"], "partials")

## Load tokenizer and WikiText-2

Accept the LLaMA-2 license and provide `HF_TOKEN` (Colab secret or env var). Tokenization and
non-overlapping 2048-token blocking match `bfp.ipynb`. `MAX_PROFILE_BLOCKS` complete blocks
are selected at evenly spaced positions across the full test split and reused for every BFP
format. The exact block indices are stored in each JSON file.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)

dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
seq_len = input_ids.size(1)
usable = seq_len // CONTEXT_LENGTH * CONTEXT_LENGTH
total_blocks = usable // CONTEXT_LENGTH
profile_blocks = min(MAX_PROFILE_BLOCKS, total_blocks)
profile_block_indices = evenly_spaced_indices(total_blocks, profile_blocks).tolist()
print(
    f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}; "
    f"full blocks: {total_blocks}; profiling blocks: {profile_block_indices}"
)
assert len(profile_block_indices) == profile_blocks
assert len(set(profile_block_indices)) == profile_blocks

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_paths = []
summary = []

for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(BFP, mantissa_bits=mantissa_bits)
    config.validate()
    bfp_bits = 1 + config.mantissa_bits
    fmt = f"BFP{bfp_bits}"
    output_path = OUTPUT_DIR / f"bfp{bfp_bits}-g{config.block_size}.json"

    print(f"\n{'=' * 72}\nProfiling {fmt}: {config}\n{'=' * 72}")
    torch.manual_seed(0)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False

    quantized_layers = replace_linear_layers(model, config)
    assert len(quantized_layers) > 0
    torch.cuda.empty_cache()

    profiler = ExponentProfiler(
        config, EXP_BIN_MIN, EXP_BIN_MAX,
        token_stride=TOKEN_STRIDE, out_subsample=OUT_SUBSAMPLE, group_chunk=GROUP_CHUNK,
    )
    profiler.attach(model)

    device = next(model.parameters()).device
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    with torch.inference_mode():
        for block_index in tqdm(profile_block_indices, desc=f"Profiling {fmt}"):
            begin = block_index * CONTEXT_LENGTH
            batch = input_ids[:, begin:begin + CONTEXT_LENGTH].to(device)
            model(batch, use_cache=False)
    torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - start
    peak_memory_gib = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    profiler.remove()

    records = profiler.export()
    if len(records) != len(quantized_layers):
        raise RuntimeError("Not every quantized Linear layer was profiled.")
    if any(record["num_calls"] != profile_blocks for record in records):
        raise RuntimeError("Unexpected per-layer profiling call count.")
    if any(record["num_underflow"] or record["num_overflow"] for record in records):
        raise RuntimeError("Exponent guard bins are nonzero; widen the histogram range.")
    payload = {
        "metadata": {
            "model": MODEL_ID,
            "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
            "split": SPLIT,
            "analysis": f"per-layer Group-{config.block_size} partial-sum exponent histogram",
            "path": "baseline BFP (no OB-Skip)",
            "format": f"{fmt} (1S{config.mantissa_bits}M + shared E{config.shared_exponent_bits})",
            "bfp_config": asdict(config),
            "threshold_independent": True,
            "evaluation_protocol": EVALUATION_PROTOCOL,
            "context_length": CONTEXT_LENGTH,
            "stride": STRIDE,
            "drop_remainder": DROP_REMAINDER,
            "source_input_tokens": int(input_ids.numel()),
            "complete_blocks": total_blocks,
            "profile_blocks": profile_blocks,
            "profiled_input_tokens": profile_blocks * CONTEXT_LENGTH,
            "profile_block_indices": profile_block_indices,
            "block_sampling": "evenly_spaced_across_complete_blocks",
            "token_stride": TOKEN_STRIDE,
            "out_subsample": OUT_SUBSAMPLE,
            "output_channel_sampling": "all_if_within_cap_else_evenly_spaced_inclusive",
            "exp_bin_min": EXP_BIN_MIN,
            "exp_bin_max": EXP_BIN_MAX,
            "histogram_semantics": "counts[i] corresponds to exponent exp_bin_min + i; zeros excluded",
            "quantized_linear_layers": len(quantized_layers),
            "elapsed_seconds": elapsed,
            "peak_gpu_memory_gib": peak_memory_gib,
            "gpu": torch.cuda.get_device_name(0),
            "cuda": torch.version.cuda,
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "transformers": transformers.__version__,
            "datasets": datasets.__version__,
        },
        "layers": records,
    }
    output_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    output_paths.append(output_path)
    total_partials = sum(record["total_partials"] for record in records)
    summary.append((fmt, len(records), total_partials, elapsed))
    print(
        f"Saved {output_path.resolve()} | layers={len(records)} | "
        f"partials={total_partials:,} | {elapsed:.1f}s"
    )

    del model, quantized_layers, profiler
    gc.collect()
    torch.cuda.empty_cache()

print("\nProfiling complete:")
for fmt, nlayers, npart, el in summary:
    print(f"  {fmt}: {nlayers} layers, {npart:,} partials, {el:.1f}s")

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=path.name)

print(f"Created: {ARCHIVE_PATH.resolve()}")
print("Unzip the JSON files into: results/profiling/exponent/llama2-7b/")
try:
    from google.colab import files
    files.download(str(ARCHIVE_PATH))
except Exception as exc:
    print(f"Manual download needed ({exc}). Grab {ARCHIVE_PATH} from the file browser.")